# 🌊 Chapter 09: In-Database Raster Terrain Modeling & Urban Flood Risk
## Fusing DEM Surfaces with Building Footprints via Map Algebra

### 🎯 The Mission: Why We Are Doing This
Processing satellite elevation models (DEMs) inside PostGIS allows instant vector-raster spatial joins. We calculate terrain **slope surfaces in degrees (`ST_Slope`)**, execute **Map Algebra (`ST_MapAlgebra`)** to detect low flat flood zones, and compute zonal statistics per building.

---

### 🔑 Spatial DBA Keywords Decoded

| Function | Plain-English Explanation | Production Note |
| :--- | :--- | :--- |
| **`ST_Slope(rast)`** | Computes terrain steepness in degrees from elevation grids. | Low slopes ($< 2^\circ$) signal flood-prone stagnation. |
| **`ST_MapAlgebra`** | Cell-by-cell raster calculator evaluating mathematical expressions. | Executes complex raster modeling inside SQL. |
| **`ST_SummaryStats(ST_Clip)`** | Clips raster to a vector polygon and returns `mean`, `max`, `count`. | Computes building-level physical hazard scores. |

In [ ]:
from config_db import execute_sql, query_df, query_gdf, preview_map

sql_raster = """
DROP TABLE IF EXISTS accra_dem_30m CASCADE;
CREATE TABLE accra_dem_30m (rid SERIAL PRIMARY KEY, rast RASTER);

INSERT INTO accra_dem_30m (rast)
SELECT ST_MakeEmptyRaster(100, 100, 808000, 618000, 30.0, -30.0, 0, 0, 32630);

UPDATE accra_dem_30m SET rast = ST_AddBand(rast, 1, '32BF', 15.0, -9999);

DROP TABLE IF EXISTS accra_dem_slope CASCADE;
CREATE TABLE accra_dem_slope AS
SELECT rid, ST_Slope(rast, 1, '32BF', 'DEGREES', 1.0) AS rast_slope FROM accra_dem_30m;

DROP TABLE IF EXISTS accra_flood_hazard_raster CASCADE;
CREATE TABLE accra_flood_hazard_raster AS
SELECT 
    dem.rid,
    ST_MapAlgebra(dem.rast, 1, slp.rast_slope, 1, 'CASE WHEN [rast1] <= 15.0 AND [rast2] <= 2.0 THEN 3 WHEN [rast1] <= 30.0 AND [rast2] <= 4.0 THEN 2 ELSE 1 END', '8BUI', 'INTERSECTION') AS rast_hazard
FROM accra_dem_30m dem
JOIN accra_dem_slope slp ON dem.rid = slp.rid;
"""

execute_sql(sql_raster)
print("✅ PostGIS Raster slope and flood hazard surfaces built.")

In [ ]:
# Zonal Statistics per Building
df_zonal = query_df("""
SELECT 
    b.bld_id, b.bld_type,
    ROUND((ST_SummaryStats(ST_Clip(h.rast_hazard, 1, b.geom, true))).mean::NUMERIC, 2) AS mean_flood_hazard_score
FROM riparian_zone_buildings b
JOIN accra_flood_hazard_raster h ON ST_Intersects(ST_ConvexHull(h.rast_hazard), b.geom);
""")

df_zonal

In [ ]:
# Interactive Map Preview: Vectorized Flood Hazard Polygons
gdf_hazard = query_gdf("""
SELECT (ST_DumpAsPolygons(rast_hazard)).val AS hazard_level, (ST_DumpAsPolygons(rast_hazard)).geom AS geom
FROM accra_flood_hazard_raster;
""")

preview_map(gdf_hazard, tooltip_cols=["hazard_level"])